# Imports

In [134]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [135]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), os.pardir))

In [136]:
from Tools.leica_tools import RawLoader, parse_lif
from Tools.sample_tools import Sample
from Tools.db_tools import DbManager

# ---- for debugging ----
import numpy as np
import pandas as pd
import os
import glob
from readlif.reader import LifFile
import datetime
from dotenv import load_dotenv


In [137]:
parse_lif('/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TECH_XG_009.lif')

,index,name,timestamp,t_index,n_channels,bit_depth,resolution,merged
0,0,TileScan 1/oil_1_300_170 Merged,0,0,4,16,1.527160,True
1,0,TileScan 1/oil_1_300_170 Merged,1,1,4,16,1.527160,True
2,0,TileScan 1/oil_1_300_170 Merged,2,2,4,16,1.527160,True
3,0,TileScan 1/oil_1_300_170 Merged,3,3,4,16,1.527160,True
4,0,TileScan 1/oil_1_300_170 Merged,4,4,4,16,1.527160,True
5,0,TileScan 1/oil_1_300_170 Merged,5,5,4,16,1.527160,True
6,0,TileScan 1/oil_1_300_170 Merged,6,6,4,16,1.527160,True
7,0,TileScan 1/oil_1_300_170 Merged,7,7,4,16,1.527160,True
8,0,TileScan 1/oil_1_300_170 Merged,8,8,4,16,1.527160,True
9,1,TileScan 1/oil_2_300_170 Merged,0,0,4,16,1.527159,True


# Data prep

In [138]:
expID = 'TECH_XG_009'
rawloader = RawLoader(expID)
rawloader.frame_df

,droplet_size,size_range,image_index,t_index,time,condition,path
frameID,,,,,,,
0,90,5,0,0,1,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
1,90,5,0,1,2,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
2,90,5,2,0,3,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
3,90,5,2,1,4,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
4,90,5,2,2,5,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
5,90,5,2,3,6,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
6,90,5,2,4,7,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
7,90,5,2,5,8,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
8,90,5,2,6,9,HFE7500_THP1_300_170,/Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...


# Droplet detection

Execute a preview run of the droplet detection. An image will be saved to the exp folder in the analyses directory. 
If droplets are not well detected consider changing the droplet size estimate in setup.xlsx (re-run RawLoader API).

In [139]:
frameID = 2
sample = Sample(expID, frameID)
sample.detect_droplets(mode='sweep')
sample.visualize_droplets(channel=0,save=True)

3317 droplets in frame 2 detected 



Run droplet detection through all frames of the experiment. drop_register.csv will be created at the end of the process.

In [131]:
rawloader = RawLoader(expID)
df = rawloader.frame_df.copy()

conditions = ["HFE7500_THP1_300_170", "RAN101_THP1_300_170"]
df = df[df["condition"].isin(conditions)].copy()
df = df.sort_index()  # sort by the index, which is frameID

# print(df[["image_index", "t_index", "time", "condition", "path"]])

all_dfs = []
for frameID in df.index.astype(int):
    print(f"Processing frameID: {frameID}")
    sample = Sample(expID, int(frameID))
    all_dfs.append(sample.detect_droplets(mode="sweep", return_df=True))

droplets = pd.concat(all_dfs, ignore_index=True).reset_index(drop=True).rename_axis("GlobalID")

rawloader.update_droplet_df(droplets)
print("Saved droplets.csv to:", os.path.join(rawloader.exp_dir, "droplets.csv"))

Processing frameID: 0
droplet_size                                                   90
size_range                                                      5
image_index                                                     0
t_index                                                         0
time                                                            1
condition                                    HFE7500_THP1_300_170
path            /Users/xiangxigao/Desktop/DMi8/TECH_XG_009/TEC...
frameID                                                         0
scale                                                     1.52716
bit_depth                                                      16
channels                                                        4
Name: 0, dtype: object
3316 droplets in frame 0 detected 

Processing frameID: 1
droplet_size                                                   90
size_range                                                      5
image_index                            

ValueError: There are not that many images!

In [140]:
dbm = DbManager()
dbm.detect_droplets(expID, mode='sweep')

3316 droplets in frame 0 detected 

3318 droplets in frame 1 detected 

3317 droplets in frame 2 detected 

3322 droplets in frame 3 detected 

3341 droplets in frame 4 detected 

3435 droplets in frame 5 detected 

3443 droplets in frame 6 detected 

3448 droplets in frame 7 detected 

3494 droplets in frame 8 detected 

3500 droplets in frame 9 detected 

3500 droplets in frame 10 detected 

3182 droplets in frame 11 detected 

2875 droplets in frame 12 detected 

2802 droplets in frame 13 detected 

2846 droplets in frame 14 detected 

2846 droplets in frame 15 detected 

2862 droplets in frame 16 detected 

2881 droplets in frame 17 detected 

2879 droplets in frame 18 detected 

2898 droplets in frame 19 detected 

2910 droplets in frame 20 detected 

2911 droplets in frame 21 detected 

2908 droplets in frame 22 detected 

2914 droplets in frame 23 detected 



# Outlier detection

In [141]:
dbm = DbManager()
dbm.detect_outliers(expID, model_name='outlier_v3.h5')

2349/2349 ━━━━━━━━━━━━━━━━━━━━ 109s 46ms/step


/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)
2026-09-09 18:02:01.861758: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-09 18:02:08.117437: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-09 18:02:08.735500: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-09 18:02:08.823153: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-09 18:02:08.905840: W tensorflow/core/fr

In [22]:
sample = Sample(expID, 0)
sample.reload_droplets()
sample.visualize_droplets(channel=0)

# Workpackage Generation

In [3]:
dbm = DbManager()
dbm.generate_wp(expID='NKIP_FA_065', exclude_query='outlier == True')

2024-09-08 22:10:21.239744: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.339381: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.549281: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.961074: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:22.772556: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# Cell counting

In [142]:
dbm = DbManager()
dbm.cell_count(expID='TECH_XG_009', model_name='cell_count_v3.h5')

2026-09-09 18:02:21.146003: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2349/2349 ━━━━━━━━━━━━━━━━━━━━ 375s 160ms/step


/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)
2026-09-09 18:08:36.276213: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
